"""
01_efd_analysis
Elliptic Fourier Descriptor (EFD) features, PCA, PERMANOVA/PERMDISP for the
A. marina leaf outlines (n = 442).
 
Inputs : all_outlines_cleaned_442   (one closed contour per leaf; file name starts with the country code)
Outputs: 77harmonics_normalized.csv       (leaf_id, 77 EFD coefficients, country)
         tableIII_efd_pca_variance.csv
         efd_centroid_distances.csv
         efd_deformation_PC1.png ... PC3.png
"""

In [1]:
import os
 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyefd import elliptic_fourier_descriptors, reconstruct_contour
from scipy.spatial.distance import pdist, squareform
from skbio.stats.distance import DistanceMatrix, permanova, permdisp
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
 
OUTLINE_ROOT = "all_outlines_cleaned_442"
HARMONICS = 20
SEED = 42

In [2]:
# ---------------------------------------------------------------- 1. load outlines
outlines, countries, leaf_ids = [], [], []
for fname in sorted(os.listdir(OUTLINE_ROOT)):
    if not fname.endswith(".npy"):
        continue
    outlines.append(np.load(os.path.join(OUTLINE_ROOT, fname)).astype(np.float64))
    countries.append(fname.split("_")[0])          # country code = first token of the file name
    leaf_ids.append(fname.replace(".npy", ""))     # same stem as the shape-feature file name (minus ".png")
 
assert len(outlines) == 442, f"expected 442 outlines, found {len(outlines)}"
print("Outlines loaded:", len(outlines), "| countries:", sorted(set(countries)))

Outlines loaded: 442 | countries: ['BH', 'OM', 'QA', 'SA', 'UAE']


In [3]:
# ---------------------------------------------------------------- 2. EFD (20 harmonics, normalised)
efd = pd.DataFrame(
    [elliptic_fourier_descriptors(c, order=HARMONICS, normalize=True).flatten() for c in outlines],
    index=leaf_ids,
)
# drop the three invariant coefficients of the normalised first harmonic (a1 = 1, b1 = 0, c1 = 0)
efd = efd.drop(columns=[0, 1, 2])
assert efd.shape[1] == 77
efd_out = efd.copy()
efd_out["country"] = countries
efd_out.to_csv("77harmonics_normalized.csv", index_label="leaf_id")

In [4]:
# ---------------------------------------------------------------- 3. PCA
efd_scaler = StandardScaler()
X_scaled = efd_scaler.fit_transform(efd)
pca_efd = PCA()
X_pca = pca_efd.fit_transform(X_scaled)
 
explained = pca_efd.explained_variance_ratio_ * 100
variance_table = pd.DataFrame(
    {"PC": [f"PC{i + 1}" for i in range(len(explained))],
     "Variance (%)": explained,
     "Cumulative (%)": np.cumsum(explained)}
).round(2)
variance_table.head(10).to_csv("tableIII_efd_pca_variance.csv", index=False)
print(variance_table.head(10).to_string(index=False))

  PC  Variance (%)  Cumulative (%)
 PC1          8.51            8.51
 PC2          7.23           15.74
 PC3          6.93           22.67
 PC4          5.51           28.18
 PC5          4.29           32.47
 PC6          3.49           35.96
 PC7          3.10           39.07
 PC8          3.00           42.07
 PC9          2.84           44.91
PC10          2.68           47.59


In [5]:
# ---------------------------------------------------------------- 4. outline deformation along PC1-PC3
def reconstruct_shape(pc_index, sd_multiplier):
    """Outline at +/- k SD along one PC (all other PCs at the mean)."""
    pc_vector = np.zeros(pca_efd.n_components_)
    pc_vector[pc_index] = sd_multiplier * np.sqrt(pca_efd.explained_variance_[pc_index])
    coeffs_reduced = efd_scaler.inverse_transform([pca_efd.inverse_transform(pc_vector)])[0]
    # restore the three normalised constants of harmonic 1: a1 = 1, b1 = 0, c1 = 0
    coeffs_full = np.concatenate([[1.0, 0.0, 0.0], coeffs_reduced]).reshape((HARMONICS, 4))
    return reconstruct_contour(coeffs_full, locus=(0, 0), num_points=400)
 
 
for pc in range(3):
    plt.figure(figsize=(6, 6))
    for k, style, label in [(0, "-", "Mean"), (2, "--", f"+2 SD PC{pc + 1}"), (-2, ":", f"-2 SD PC{pc + 1}")]:
        shape = reconstruct_shape(pc, k)
        plt.plot(shape[:, 0], shape[:, 1], linestyle=style, label=label)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.legend()
    plt.title(f"Outline Deformation Along PC{pc + 1} (EFD)")
    plt.savefig(f"efd_deformation_PC{pc + 1}.png", dpi=300, bbox_inches="tight")
    plt.close()

In [6]:
# ---------------------------------------------------------------- 5. country centroids (PC1-PC3)
pca_df = pd.DataFrame(X_pca[:, :3], columns=["PC1", "PC2", "PC3"], index=leaf_ids)
pca_df["country"] = countries
centroids = pca_df.groupby("country")[["PC1", "PC2", "PC3"]].mean()
centroid_dist = pd.DataFrame(squareform(pdist(centroids)), index=centroids.index, columns=centroids.index)
centroid_dist.round(2).to_csv("efd_centroid_distances.csv")
print(centroid_dist.round(2))

country    BH    OM    QA    SA   UAE
country                              
BH       0.00  1.16  1.64  2.26  1.46
OM       1.16  0.00  0.72  1.12  0.35
QA       1.64  0.72  0.00  0.80  0.50
SA       2.26  1.12  0.80  0.00  0.80
UAE      1.46  0.35  0.50  0.80  0.00


In [7]:
# ---------------------------------------------------------------- 6. PERMANOVA / PERMDISP in EFD space
ids = [str(i) for i in range(len(leaf_ids))]
dm = DistanceMatrix(squareform(pdist(X_scaled)), ids=ids)
grouping = pd.Series(countries, index=ids, name="country")
 
res = permanova(dm, grouping, permutations=999, seed=SEED)
F, N, k = res["test statistic"], len(grouping), grouping.nunique()
R2 = F * (k - 1) / (F * (k - 1) + (N - k))            # scikit-bio does not return R2
print(f"EFD PERMANOVA: pseudo-F = {F:.2f}, p = {res['p-value']:.3f}, R2 = {R2:.4f}")
print("EFD PERMDISP:\n", permdisp(dm, grouping, permutations=999, seed=SEED))

EFD PERMANOVA: pseudo-F = 1.89, p = 0.001, R2 = 0.0170


/Users/mustafamuhamed/morpho_env/lib/python3.10/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:164: RuntimeWarning: EIGH: since no value for dimensions is specified, PCoA for all dimensions will be computed, which may result in long computation time if the original distance matrix is large.
  warn(


EFD PERMDISP:
 method name                PERMDISP
test statistic name         F-value
sample size                     442
number of groups                  5
test statistic            14.404413
p-value                       0.001
number of permutations          999
Name: PERMDISP results, dtype: object
